# RIS OGD API v2.6 — Court Decision Scraper

**Purpose:** Query Austrian court decisions (Judikatur) via the official RIS Open Government Data API for keywords related to parental alienation and child welfare.

**API Base:** `https://data.bka.gv.at/ris/api/v2.6/`  
**Documentation:** [API Handbook (PDF)](https://data.bka.gv.at/ris/ogd/v2.6/Documents/Dokumentation_OGD-RIS_API.pdf)  
**Data catalog:** [data.gv.at](https://www.data.gv.at/katalog/dataset/ris2_6)  

**Keywords:** `Entfremdung`, `Kindeswohl`, `elterliche Entfremdung`, `Kontaktrecht`, `Obsorge`

---

### Verified API Response Structure
Based on live testing, the JSON response has this layout:
```
OgdSearchResult
  └─ OgdDocumentResults
       ├─ Hits: { @pageNumber, @pageSize, #text: "189" }  ← total hits
       └─ OgdDocumentReference: [                          ← array of results
            └─ Data
                 ├─ Metadaten
                 │    ├─ Technisch: { ID, Applikation, Organ }
                 │    ├─ Allgemein: { Veroeffentlicht, Geaendert, DokumentUrl }
                 │    └─ Judikatur
                 │         ├─ Dokumenttyp ("Rechtssatz" | "Text")
                 │         ├─ Geschaeftszahl.item (string of case numbers)
                 │         ├─ Normen.item (string or list)
                 │         ├─ Entscheidungsdatum
                 │         ├─ EuropeanCaseLawIdentifier (ECLI)
                 │         └─ Justiz
                 │              ├─ Gericht ("OGH", "OLG", etc.)
                 │              ├─ Rechtsgebiete.item
                 │              ├─ Rechtssatznummern.item
                 │              └─ Entscheidungstexte.item[]
                 └─ Dokumentliste
                      └─ ContentReference
                           └─ Urls.ContentUrl[]
                                └─ { DataType: "Xml"|"Html"|"Rtf"|"Pdf", Url }
```

**Important:** Full decision text is NOT in search results. You must fetch it from the `ContentUrl` links (Html or Xml).

The API defaults to pageSize=20 regardless of `Seitengroesse`. The `Seite` parameter controls pagination.

## 0. Setup

In [1]:
#!pip install bs4
#!pip install lxml

In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re
import json
import logging
from datetime import datetime
from pathlib import Path
from urllib.parse import urlencode
from xml.etree import ElementTree as ET

# --- Configuration ---
API_BASE = "https://data.bka.gv.at/ris/api/v2.6"

KEYWORDS = [
    "Entfremdung",
   # "Kindeswohl",
   # "elterliche Entfremdung",
   # "Kontaktrecht",
   # "Obsorge",
]

# Which court app to query
# "Justiz" = OGH, OLG, LG, BG (most relevant for family law)
APPLICATIONS = ["Justiz"]

# API returns 20 per page (seems fixed regardless of Seitengroesse)
PAGE_SIZE = 20

# Polite delay between requests (seconds)
REQUEST_DELAY = 1.5

# Output directory
OUTPUT_DIR = Path("../data/ris")
OUTPUT_DIR.mkdir(exist_ok=True)

# Logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s — %(levelname)s — %(message)s")
logger = logging.getLogger(__name__)

print(f"API Base: {API_BASE}")
print(f"Output: {OUTPUT_DIR.resolve()}")
print(f"Keywords: {KEYWORDS}")

/Users/maksimsmirnov/Desktop/thesis/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


API Base: https://data.bka.gv.at/ris/api/v2.6
Output: /Users/maksimsmirnov/Desktop/thesis/data/ris
Keywords: ['Entfremdung']


In [3]:
# --- HTTP Session ---
session = requests.Session()
session.headers.update({
    "Accept": "application/json",
    "User-Agent": "MasterThesis-RIS-Research/1.0 (Academic Research)",
})


def api_get(endpoint, params, retries=3):
    """GET request to RIS API with retry logic. Returns parsed JSON or None."""
    url = f"{API_BASE}/{endpoint}"
    for attempt in range(retries):
        try:
            time.sleep(REQUEST_DELAY)
            resp = session.get(url, params=params, timeout=30)
            resp.raise_for_status()
            return resp.json()
        except requests.exceptions.HTTPError as e:
            logger.warning(f"HTTP {resp.status_code} attempt {attempt+1}: {e}")
            if resp.status_code == 429:
                time.sleep(10 * (attempt + 1))
            elif resp.status_code >= 500:
                time.sleep(5 * (attempt + 1))
            else:
                return None
        except requests.exceptions.RequestException as e:
            logger.warning(f"Request error attempt {attempt+1}: {e}")
            time.sleep(5 * (attempt + 1))
        except ValueError:
            logger.warning(f"JSON parse error. Raw: {resp.text[:300]}")
            return None
    logger.error(f"All retries failed for {url}")
    return None


def fetch_url(url, retries=3):
    """Fetch any URL with retry. Returns response text or None."""
    for attempt in range(retries):
        try:
            time.sleep(REQUEST_DELAY)
            resp = session.get(url, timeout=30)
            resp.raise_for_status()
            resp.encoding = "utf-8"
            return resp.text
        except Exception as e:
            logger.warning(f"Fetch attempt {attempt+1} failed for {url[:80]}: {e}")
            if attempt < retries - 1:
                time.sleep(3 * (attempt + 1))
    return None


print("Session ready.")

Session ready.


## 1. Response Parser

These functions match the verified response structure from live testing.

In [4]:
def parse_api_response(data):
    """
    Parse the RIS API JSON response.
    Returns (total_hits: int, documents: list[dict]).
    
    Verified structure:
      data["OgdSearchResult"]["OgdDocumentResults"]["Hits"]["#text"] = total
      data["OgdSearchResult"]["OgdDocumentResults"]["OgdDocumentReference"] = docs[]
    """
    if not data:
        return 0, []
    
    try:
        results = data["OgdSearchResult"]["OgdDocumentResults"]
        
        # Total hits
        hits_obj = results.get("Hits", {})
        total_hits = int(hits_obj.get("#text", 0))
        
        # Documents
        docs = results.get("OgdDocumentReference", [])
        if isinstance(docs, dict):  # single result comes as dict, not list
            docs = [docs]
        
        return total_hits, docs
    
    except (KeyError, TypeError) as e:
        logger.warning(f"Parse error: {e}")
        return 0, []


def extract_document_fields(doc):
    """
    Extract fields from a single OgdDocumentReference.
    
    Verified paths:
      Data.Metadaten.Technisch.{ID, Applikation, Organ}
      Data.Metadaten.Allgemein.{DokumentUrl, Veroeffentlicht, Geaendert}
      Data.Metadaten.Judikatur.{Dokumenttyp, Geschaeftszahl.item, Normen.item,
          Entscheidungsdatum, EuropeanCaseLawIdentifier}
      Data.Metadaten.Judikatur.Justiz.{Gericht, Rechtsgebiete.item,
          Rechtssatznummern.item, Entscheidungstexte.item[]}
      Data.Dokumentliste.ContentReference.Urls.ContentUrl[]
    """
    if not doc or not isinstance(doc, dict):
        return {}
    
    data = doc.get("Data", {})
    meta = data.get("Metadaten", {})
    tech = meta.get("Technisch", {})
    allg = meta.get("Allgemein", {})
    jud = meta.get("Judikatur", {})
    justiz = jud.get("Justiz", {})
    
    # Helper for .item fields (can be string, list, or dict)
    def get_item(d, key, default=""):
        """Safely get a .item value that may be str, list, or nested."""
        val = d.get(key, {})
        if isinstance(val, str):
            return val
        if isinstance(val, dict):
            item = val.get("item", default)
            if isinstance(item, list):
                return "; ".join(str(i) for i in item)
            return str(item) if item else default
        if isinstance(val, list):
            return "; ".join(str(i) for i in val)
        return default
    
    # --- Extract content URLs ---
    content_urls = {}
    try:
        docliste = data.get("Dokumentliste", {})
        content_ref = docliste.get("ContentReference", {})
        if isinstance(content_ref, list):
            content_ref = content_ref[0] if content_ref else {}
        urls_section = content_ref.get("Urls", {})
        url_list = urls_section.get("ContentUrl", [])
        if isinstance(url_list, dict):
            url_list = [url_list]
        for u in url_list:
            if isinstance(u, dict):
                content_urls[u.get("DataType", "")] = u.get("Url", "")
    except (KeyError, TypeError, AttributeError):
        pass
    
    # --- Extract Entscheidungstexte links ---
    entscheidungstexte = []
    et_raw = justiz.get("Entscheidungstexte", {})
    if isinstance(et_raw, dict):
        items = et_raw.get("item", [])
        if isinstance(items, dict):
            items = [items]
        if isinstance(items, list):
            entscheidungstexte = items
    
    return {
        # Technical
        "id": tech.get("ID", ""),
        "applikation": tech.get("Applikation", ""),
        "organ": tech.get("Organ", ""),
        # General
        "dokument_url": allg.get("DokumentUrl", ""),
        "veroeffentlicht": allg.get("Veroeffentlicht", ""),
        "geaendert": allg.get("Geaendert", ""),
        # Judikatur
        "dokumenttyp": jud.get("Dokumenttyp", ""),
        "geschaeftszahl": get_item(jud, "Geschaeftszahl"),
        "normen": get_item(jud, "Normen"),
        "entscheidungsdatum": jud.get("Entscheidungsdatum", ""),
        "ecli": jud.get("EuropeanCaseLawIdentifier", ""),
        # Justiz sub-fields
        "gericht": justiz.get("Gericht", ""),
        "rechtsgebiete": get_item(justiz, "Rechtsgebiete"),
        "rechtssatznummern": get_item(justiz, "Rechtssatznummern"),
        # Content URLs for full text fetching
        "content_url_html": content_urls.get("Html", ""),
        "content_url_xml": content_urls.get("Xml", ""),
        "content_url_pdf": content_urls.get("Pdf", ""),
        # Number of linked Entscheidungstexte
        "n_entscheidungstexte": len(entscheidungstexte),
    }


print("Parser functions ready.")

Parser functions ready.


## 2. Quick Test

In [5]:
# --- Test query ---
test_params = {
    "Applikation": "Justiz",
    "Suchworte": "Kindeswohl",
    "Seite": 1,
}

test_data = api_get("Judikatur", test_params)
total, docs = parse_api_response(test_data)

print(f"Total hits: {total}")
print(f"Documents on page 1: {len(docs)}")

if docs:
    parsed = extract_document_fields(docs[0])
    print(f"\nFirst document:")
    for k, v in parsed.items():
        display = str(v)[:120] + "..." if isinstance(v, str) and len(v) > 120 else v
        print(f"  {k}: {display}")

2026-03-17 13:46:32,788 — WARNING — Request error attempt 1: HTTPSConnectionPool(host='data.bka.gv.at', port=443): Max retries exceeded with url: /ris/api/v2.6/Judikatur?Applikation=Justiz&Suchworte=Kindeswohl&Seite=1 (Caused by NameResolutionError("HTTPSConnection(host='data.bka.gv.at', port=443): Failed to resolve 'data.bka.gv.at' ([Errno 8] nodename nor servname provided, or not known)"))
2026-03-17 13:46:39,296 — WARNING — Request error attempt 2: HTTPSConnectionPool(host='data.bka.gv.at', port=443): Max retries exceeded with url: /ris/api/v2.6/Judikatur?Applikation=Justiz&Suchworte=Kindeswohl&Seite=1 (Caused by NameResolutionError("HTTPSConnection(host='data.bka.gv.at', port=443): Failed to resolve 'data.bka.gv.at' ([Errno 8] nodename nor servname provided, or not known)"))


Total hits: 189
Documents on page 1: 20

First document:
  id: JJR_20130508_OGH0002_0060OB00041_13T0000_004
  applikation: Justiz
  organ: OGH
  dokument_url: https://www.ris.bka.gv.at/Dokument.wxe?Abfrage=Justiz&Dokumentnummer=JJR_20130508_OGH0002_0060OB00041_13T0000_004
  veroeffentlicht: 2013-07-09
  geaendert: 2026-02-24
  dokumenttyp: Rechtssatz
  geschaeftszahl: 6Ob41/13t; 4Ob32/13d; 6Ob74/13w; 4Ob58/13b; 1Ob126/13f; 6Ob155/13g; 3Ob145/13i; 3Ob103/13p; 7Ob211/13z; 5Ob227/13p; 1Ob2...
  normen: ABGB idF KindNamRÄG 2013 §180
  entscheidungsdatum: 2026-01-27
  ecli: ECLI:AT:OGH0002:2013:RS0128812
  gericht: OGH
  rechtsgebiete: Zivilrecht
  rechtssatznummern: RS0128812
  content_url_html: https://www.ris.bka.gv.at/Dokumente/Justiz/JJR_20130508_OGH0002_0060OB00041_13T0000_004/JJR_20130508_OGH0002_0060OB00041...
  content_url_xml: https://www.ris.bka.gv.at/Dokumente/Justiz/JJR_20130508_OGH0002_0060OB00041_13T0000_004/JJR_20130508_OGH0002_0060OB00041...
  content_url_pdf: https://www.r

In [6]:
# --- Test full-text fetch from content URL ---
# The search results DON'T include full text.
# We need to fetch the Html or Xml content URL.

if docs:
    sample = extract_document_fields(docs[0])
    
    # Prefer HTML for easier text extraction
    html_url = sample.get("content_url_html", "")
    xml_url = sample.get("content_url_xml", "")
    
    if html_url:
        print(f"Fetching HTML: {html_url[:100]}...\n")
        html_content = fetch_url(html_url)
        if html_content:
            soup = BeautifulSoup(html_content, "html.parser")
            text = soup.get_text("\n", strip=True)
            print(f"HTML text length: {len(text)} chars")
            print(f"\nPreview:\n{text[:800]}")
    elif xml_url:
        print(f"Fetching XML: {xml_url[:100]}...\n")
        xml_content = fetch_url(xml_url)
        if xml_content:
            root = ET.fromstring(xml_content)
            text = ET.tostring(root, encoding="unicode", method="text")
            print(f"XML text length: {len(text)} chars")
            print(f"\nPreview:\n{text[:800]}")
    else:
        print("No content URL found in this document.")
        print("This may be a Rechtssatz (legal principle summary) without a standalone text.")
        print("Try fetching the DokumentUrl instead:")
        print(f"  {sample.get('dokument_url', '')}")

Fetching HTML: https://www.ris.bka.gv.at/Dokumente/Justiz/JJR_20130508_OGH0002_0060OB00041_13T0000_004/JJR_20130508...

HTML text length: 16569 chars

Preview:
RIS Dokument
Gericht
OGH
Rechtssatznummer
RS0128812
Entscheidungsdatum
27.01.2026
Geschäftszahl
6Ob41/13t; 4Ob32/13d; 6Ob74/13w; 4Ob58/13b; 1Ob126/13f; 6Ob155/13g; 3Ob145/13i; 3Ob103/13p; 7Ob211/13z; 5Ob227/13p; 1Ob220/13d; 10Ob53/13m; 7Ob64/14h; 4Ob88/14s; 5Ob144/14h; 3Ob128/14s; 1Ob156/14v; 3Ob149/14d; 7Ob198/14i; 1Ob250/14t; 2Ob240/14d; 8Ob7/15k; 7Ob229/14y; 8Ob40/15p; 3Ob126/15y; 5Ob163/15d; 7Ob159/15f; 8Ob146/15a; 10Ob22/16g; 3Ob37/16m; 10Ob53/16s; 9Ob51/16i; 1Ob241/16x; 3Ob7/17a; 4Ob67/17g; 4Ob111/17b; 1Ob136/17g; 3Ob142/17d; 8Ob29/17y; 8Ob152/17m; 1Ob93/18k; 7Ob147/18w; 7Ob209/18p; 7Ob217/18i; 7Ob9/19b; 10Ob8/19b; 3Ob22/19k; 5Ob185/19w; 3Ob159/19g; 1Ob219/19s; 1Ob181/20d; 1Ob15/20t; 3Ob203/20d; 6Ob234/20k; 5Ob86/21i; 1Ob119/21p; 1Ob111/21m; 1Ob136/21p; 1Ob60/22p; 1Ob45/22g; 8Ob62/22


## 3. Full-Text Fetcher

In [7]:
def fetch_full_text(doc_record):
    """
    Fetch full decision text for a single document record.
    
    Strategy:
      1. Try content_url_html (clean text from HTML)
      2. Try content_url_xml (text from XML)
      3. Fallback: scrape dokument_url (RIS website page)
    
    Returns the extracted text string.
    """
    # Strategy 1: HTML content URL (best quality)
    html_url = doc_record.get("content_url_html", "")
    if html_url:
        raw = fetch_url(html_url)
        if raw:
            soup = BeautifulSoup(raw, "html.parser")
            text = soup.get_text("\n", strip=True)
            if len(text) > 50:
                return text
    
    # Strategy 2: XML content URL
    xml_url = doc_record.get("content_url_xml", "")
    if xml_url:
        raw = fetch_url(xml_url)
        if raw:
            try:
                root = ET.fromstring(raw)
                # Extract all text content from XML
                text_parts = []
                for elem in root.iter():
                    if elem.text and elem.text.strip():
                        text_parts.append(elem.text.strip())
                    if elem.tail and elem.tail.strip():
                        text_parts.append(elem.tail.strip())
                text = "\n".join(text_parts)
                if len(text) > 50:
                    return text
            except ET.ParseError:
                # XML might be malformed; try treating as HTML
                soup = BeautifulSoup(raw, "lxml")
                text = soup.get_text("\n", strip=True)
                if len(text) > 50:
                    return text
    
    # Strategy 3: Fallback to RIS website
    dok_url = doc_record.get("dokument_url", "")
    if dok_url:
        raw = fetch_url(dok_url)
        if raw:
            soup = BeautifulSoup(raw, "lxml")
            # Try known content containers on ris.bka.gv.at
            main = (
                soup.select_one(".contentBlock, .documentContent, #DocumentText")
                or soup.find("main")
                or soup.find("body")
            )
            if main:
                text = main.get_text("\n", strip=True)
                if len(text) > 50:
                    return text
    
    return ""


print("Full-text fetcher ready.")

Full-text fetcher ready.


## 4. Main Collection Loop

In [8]:
# --- Configuration ---

# Max results per keyword (None = all)
MAX_RESULTS_PER_KEYWORD = 500

# Date range (YYYY-MM-DD format, or None)
DATE_FROM = None  # e.g., "2000-01-01"
DATE_TO = None

# Whether to fetch full text for each document (slower but needed for NLP)
FETCH_FULL_TEXT = True

print(f"Max per keyword: {MAX_RESULTS_PER_KEYWORD}")
print(f"Date range: {DATE_FROM or 'any'} – {DATE_TO or 'any'}")
print(f"Fetch full text: {FETCH_FULL_TEXT}")

Max per keyword: 500
Date range: any – any
Fetch full text: True


In [9]:
def search_ris(keyword, applikation="Justiz", max_results=None,
               date_from=None, date_to=None):
    """
    Search RIS Judikatur API.
    Uses year-by-year windowing to work around broken pagination.
    """
    all_docs = []
    seen_ids = set()
    
    # Build year ranges to query
    start_year = int(date_from[:4]) if date_from else 1970
    end_year = int(date_to[:4]) if date_to else 2026
    
    # First: do one undated query to get total hits and first 20
    params = {"Applikation": applikation, "Suchworte": keyword}
    data = api_get("Judikatur", params)
    total_hits, docs = parse_api_response(data)
    logger.info(f"'{keyword}' ({applikation}): {total_hits} total hits")
    
    # Collect first page
    for doc in docs:
        parsed = extract_document_fields(doc)
        doc_id = parsed.get("id", "")
        if doc_id and doc_id not in seen_ids:
            seen_ids.add(doc_id)
            parsed["search_keyword"] = keyword
            all_docs.append(parsed)
    
    logger.info(f"  Undated query: {len(all_docs)} docs")
    
    if total_hits <= 20:
        return all_docs  # got everything already
    
    # Need more — query year by year
    logger.info(f"  Paginating via year windows {start_year}–{end_year}...")
    
    for year in range(start_year, end_year + 1):
        params = {
            "Applikation": applikation,
            "Suchworte": keyword,
            "EntscheidungsdatumVon": f"{year}-01-01",
            "EntscheidungsdatumBis": f"{year}-12-31",
        }
        data = api_get("Judikatur", params)
        year_total, docs = parse_api_response(data)
        
        if not docs:
            continue
        
        new_count = 0
        for doc in docs:
            parsed = extract_document_fields(doc)
            doc_id = parsed.get("id", "")
            if doc_id and doc_id not in seen_ids:
                seen_ids.add(doc_id)
                parsed["search_keyword"] = keyword
                all_docs.append(parsed)
                new_count += 1
        
        if new_count > 0:
            logger.info(f"    {year}: +{new_count} new (year had {year_total} hits, total: {len(all_docs)})")
        
        # Warn if a single year has >20 hits (we're missing some)
        if year_total > 20:
            logger.warning(f"    ⚠️ {year} has {year_total} hits but API returns max 20. "
                         f"Consider splitting into half-year windows.")
        
        if max_results and len(all_docs) >= max_results:
            all_docs = all_docs[:max_results]
            break
    
    logger.info(f"  Total collected: {len(all_docs)}")
    return all_docs

In [10]:
# DEBUG: Compare page 1 vs page 2 raw IDs
for page_num in [1, 2, 3]:
    data = api_get("Judikatur", {
        "Applikation": "Justiz",
        "Suchworte": "Entfremdung",
        "Seite": page_num,
    })
    _, docs = parse_api_response(data)
    ids = [d.get("Data", {}).get("Metadaten", {}).get("Technisch", {}).get("ID", "?") 
           for d in docs]
    print(f"\nPage {page_num}: {len(docs)} docs")
    print(f"  First 3 IDs: {ids[:3]}")
    print(f"  Last 3 IDs:  {ids[-3:]}")
    
    # Also check what @pageNumber the API reports back
    hits = data.get("OgdSearchResult", {}).get("OgdDocumentResults", {}).get("Hits", {})
    print(f"  API says: pageNumber={hits.get('@pageNumber')}, pageSize={hits.get('@pageSize')}, total={hits.get('#text')}")
'''

This will tell us whether the API is actually ignoring `Seite` (returning the same IDs every time) or if our dedup logic is too aggressive.

Also try opening this directly in your browser to compare page 1 vs page 2:
'''
#https://data.bka.gv.at/ris/api/v2.6/Judikatur?Applikation=Justiz&Suchworte=Entfremdung&Seite=1
#https://data.bka.gv.at/ris/api/v2.6/Judikatur?Applikation=Justiz&Suchworte=Entfremdung&Seite=2


Page 1: 20 docs
  First 3 IDs: ['JJR_19780712_OGH0002_0080OB00549_7800000_001', 'JJR_19600302_OGH0002_0050OB00015_6000000_001', 'JJR_19790703_OGH0002_0050OB00637_7900000_001']
  Last 3 IDs:  ['JJR_19750618_OGH0002_0110OS00059_7500000_001', 'JJR_19980730_AUSL000_000BSW22985_9300000_001', 'JJR_19900612_OGH0002_0140OS00046_9000000_003']
  API says: pageNumber=1, pageSize=20, total=36

Page 2: 20 docs
  First 3 IDs: ['JJR_19780712_OGH0002_0080OB00549_7800000_001', 'JJR_19600302_OGH0002_0050OB00015_6000000_001', 'JJR_19790703_OGH0002_0050OB00637_7900000_001']
  Last 3 IDs:  ['JJR_19750618_OGH0002_0110OS00059_7500000_001', 'JJR_19980730_AUSL000_000BSW22985_9300000_001', 'JJR_19900612_OGH0002_0140OS00046_9000000_003']
  API says: pageNumber=1, pageSize=20, total=36

Page 3: 20 docs
  First 3 IDs: ['JJR_19780712_OGH0002_0080OB00549_7800000_001', 'JJR_19600302_OGH0002_0050OB00015_6000000_001', 'JJR_19790703_OGH0002_0050OB00637_7900000_001']
  Last 3 IDs:  ['JJR_19750618_OGH0002_0110OS00059_750

'\n\nThis will tell us whether the API is actually ignoring `Seite` (returning the same IDs every time) or if our dedup logic is too aggressive.\n\nAlso try opening this directly in your browser to compare page 1 vs page 2:\n'

In [11]:
# --- 4.1 Collect search results (metadata only — fast) ---

all_results = []

for app in APPLICATIONS:
    for kw in KEYWORDS:
        logger.info(f"\n{'='*60}")
        logger.info(f"Searching: {app} / {kw}")
        logger.info(f"{'='*60}")
        results = search_ris(
            keyword=kw, applikation=app,
            max_results=MAX_RESULTS_PER_KEYWORD,
            date_from=DATE_FROM, date_to=DATE_TO,
        )
        all_results.extend(results)

# --- Deduplicate across keywords (merge keyword labels) ---
seen = {}
for r in all_results:
    key = r.get("id") or r.get("ecli") or r.get("dokument_url")
    if not key:
        seen[str(len(seen))] = r
        continue
    if key in seen:
        existing_kw = seen[key]["search_keyword"]
        if r["search_keyword"] not in existing_kw:
            seen[key]["search_keyword"] = f"{existing_kw}; {r['search_keyword']}"
    else:
        seen[key] = r

unique_results = list(seen.values())
logger.info(f"\nTotal unique documents: {len(unique_results)}")

# Save metadata
pd.DataFrame(unique_results).to_csv(
    OUTPUT_DIR / "search_results_meta.csv", index=False, encoding="utf-8-sig"
)
print(f"Saved: {OUTPUT_DIR / 'search_results_meta.csv'}")

# --- Show Dokumenttyp distribution ---
# Rechtssatz = legal principle summary; Text = full decision text
types = pd.Series([r.get("dokumenttyp", "?") for r in unique_results]).value_counts()
print(f"\nDokumenttyp distribution:")
print(types)
print("\nNote: 'Rechtssatz' = legal principle summary, 'Text' = full decision.")
print("Both are valuable — Rechtssätze are concise legal principles cited across many cases.")

2026-03-17 13:47:01,338 — INFO — 
2026-03-17 13:47:01,372 — INFO — Searching: Justiz / Entfremdung
2026-03-17 13:47:01,384 — INFO — ============================================================
2026-03-17 13:47:03,505 — INFO — 'Entfremdung' (Justiz): 36 total hits
2026-03-17 13:47:03,507 — INFO —   Undated query: 20 docs
2026-03-17 13:47:03,508 — INFO —   Paginating via year windows 1970–2026...
2026-03-17 13:47:18,340 — INFO —     1978: +1 new (year had 1 hits, total: 21)
2026-03-17 13:47:21,673 — INFO —     1980: +1 new (year had 1 hits, total: 22)
2026-03-17 13:47:26,623 — INFO —     1983: +1 new (year had 1 hits, total: 23)
2026-03-17 13:47:31,552 — INFO —     1986: +1 new (year had 1 hits, total: 24)
2026-03-17 13:47:33,225 — INFO —     1987: +1 new (year had 1 hits, total: 25)
2026-03-17 13:47:34,987 — INFO —     1988: +2 new (year had 2 hits, total: 27)
2026-03-17 13:47:38,509 — INFO —     1990: +2 new (year had 2 hits, total: 29)
2026-03-17 13:47:40,226 — INFO —     1991: +2 new

Saved: data/ris/search_results_meta.csv

Dokumenttyp distribution:
Rechtssatz    36
Name: count, dtype: int64

Note: 'Rechtssatz' = legal principle summary, 'Text' = full decision.
Both are valuable — Rechtssätze are concise legal principles cited across many cases.


In [12]:
unique_results[0]

{'id': 'JJR_19780712_OGH0002_0080OB00549_7800000_001',
 'applikation': 'Justiz',
 'organ': 'OGH',
 'dokument_url': 'https://www.ris.bka.gv.at/Dokument.wxe?Abfrage=Justiz&Dokumentnummer=JJR_19780712_OGH0002_0080OB00549_7800000_001',
 'veroeffentlicht': '1997-06-15',
 'geaendert': '2026-01-28',
 'dokumenttyp': 'Rechtssatz',
 'geschaeftszahl': '8Ob549/78; 5Ob769/78; 3Ob660/79; 1Ob717/80; 1Ob779/80; 3Ob585/80; 6Ob798/81; 5Ob733/82; 1Ob610/82; 7Ob707/83; 2Ob577/83; 8Ob545/83; 8Ob609/84; 8Ob606/84; 7Ob683/85; 3Ob555/86; 7Ob628/86; 2Ob595/87; 1Ob648/88; 4Ob626/88; 3Ob558/88; 7Ob619/89; 8Ob596/91; 1Ob504/95 (1Ob505/95); 5Ob279/01t; 6Ob171/05y; 4Ob131/06b; 8Ob73/06b; 3Ob174/06v; 9Ob35/08z; 1Ob107/09f; 8Ob59/09y; 5Ob167/09h; 2Ob19/11z; 5Ob238/11b; 7Ob117/16f; 1Ob136/17g; 5Ob219/17t; 5Ob94/19p; 8Ob57/19v; 9Ob42/19w; 1Ob58/20s; 4Ob122/21a; 8Ob1/22p; 4Ob75/23t; 4Ob51/24i; 1Ob186/25x',
 'normen': 'ABGB §142 Da; ABGB §148 A; AußStrG 2005 §110 Abs2; AußStrG 2005 §110 Abs3',
 'entscheidungsdatum': '202

In [13]:
print(f"unique_results: {len(unique_results)}")
print(f"CHECKPOINT_FILE: {CHECKPOINT_FILE}")
print(f"Checkpoint exists: {CHECKPOINT_FILE.exists()}")

unique_results: 36


NameError: name 'CHECKPOINT_FILE' is not defined

In [39]:
# --- 4.2 Fetch full texts ---
# This is the slow step. Checkpoint saves progress every 25 docs.

CHECKPOINT_FILE = OUTPUT_DIR / "decisions_checkpoint.json"

# Load checkpoint if resuming
if CHECKPOINT_FILE.exists():
    with open(CHECKPOINT_FILE, "r", encoding="utf-8") as f:
        collected = json.load(f)
    fetched_ids = {d["id"] for d in collected if d.get("id")}
    logger.info(f"Resuming from checkpoint: {len(collected)} already done")
else:
    collected = []
    fetched_ids = set()

to_process = [r for r in unique_results if r.get("id") not in fetched_ids]
logger.info(f"Documents to process: {len(to_process)}")

for i, doc_meta in enumerate(to_process):
    doc_id = doc_meta.get("id", "?")
    gz = doc_meta.get("geschaeftszahl", "")[:50]
    logger.info(f"[{i+1}/{len(to_process)}] {doc_id[:40]} ({gz})")
    
    # Fetch full text
    if FETCH_FULL_TEXT:
        full_text = fetch_full_text(doc_meta)
    else:
        full_text = ""
    
    record = {
        **doc_meta,
        "full_text": full_text,
        "text_length": len(full_text),
        "fetched_at": datetime.now().isoformat(),
    }
    collected.append(record)
    
    # Checkpoint
    if (i + 1) % 25 == 0:
        with open(CHECKPOINT_FILE, "w", encoding="utf-8") as f:
            json.dump(collected, f, ensure_ascii=False, indent=1)
        logger.info(f"  Checkpoint saved ({len(collected)} total)")

# Final save
with open(CHECKPOINT_FILE, "w", encoding="utf-8") as f:
    json.dump(collected, f, ensure_ascii=False, indent=1)

logger.info(f"\n✅ Done! {len(collected)} documents collected.")
texts_found = sum(1 for d in collected if d.get("text_length", 0) > 100)
logger.info(f"   With full text: {texts_found}")
logger.info(f"   Without: {len(collected) - texts_found}")

2026-03-07 19:29:45,546 — INFO — Resuming from checkpoint: 36 already done
2026-03-07 19:29:45,548 — INFO — Documents to process: 0
2026-03-07 19:29:45,560 — INFO — 
✅ Done! 36 documents collected.
2026-03-07 19:29:45,561 — INFO —    With full text: 36
2026-03-07 19:29:45,563 — INFO —    Without: 0


## 5. Build Analysis DataFrame

In [36]:
# --- Load & build DataFrame ---
with open(CHECKPOINT_FILE, "r", encoding="utf-8") as f:
    collected = json.load(f)

df = pd.DataFrame(collected)
print(f"Loaded {len(df)} documents")
print(f"Columns: {list(df.columns)}")

# Parse dates
df["date_parsed"] = pd.to_datetime(df["entscheidungsdatum"], errors="coerce")
df["year"] = df["date_parsed"].dt.year

# Text length
df["text_length"] = df["full_text"].fillna("").str.len()

# Keyword presence flags
for kw in KEYWORDS:
    col = f"contains_{kw.lower().replace(' ', '_')}"
    df[col] = df["full_text"].fillna("").str.contains(kw, case=False, na=False)

# Stats
print(f"\nWith full text (>100 chars): {(df['text_length'] > 100).sum()}")
print(f"Year range: {df['year'].min()} – {df['year'].max()}")
print(f"\nGericht distribution:")
print(df["gericht"].value_counts())
print(f"\nDokumenttyp:")
print(df["dokumenttyp"].value_counts())
print(f"\nSearch keywords:")
print(df["search_keyword"].str.split("; ").explode().value_counts())

Loaded 36 documents
Columns: ['id', 'applikation', 'organ', 'dokument_url', 'veroeffentlicht', 'geaendert', 'dokumenttyp', 'geschaeftszahl', 'normen', 'entscheidungsdatum', 'ecli', 'gericht', 'rechtsgebiete', 'rechtssatznummern', 'content_url_html', 'content_url_xml', 'content_url_pdf', 'n_entscheidungstexte', 'search_keyword', 'full_text', 'text_length', 'fetched_at']

With full text (>100 chars): 36
Year range: 1978 – 2025

Gericht distribution:
gericht
OGH          35
AUSL EGMR     1
Name: count, dtype: int64

Dokumenttyp:
dokumenttyp
Rechtssatz    36
Name: count, dtype: int64

Search keywords:
search_keyword
Entfremdung    36
Name: count, dtype: int64


In [37]:
# --- Extract referenced legal norms from text ---

def extract_norms_from_text(text):
    """Extract norm citations like '§ 180 ABGB', 'Art 8 EMRK'."""
    if not text:
        return []
    pattern = r"(?:§§?|Art\.?)\s*\d+[a-z]?(?:\s*(?:bis|-)\s*\d+[a-z]?)?\s+[A-ZÄÖÜa-zäöü]+"
    return list(set(re.findall(pattern, text)))

df["extracted_norms"] = df["full_text"].fillna("").apply(extract_norms_from_text)
df["extracted_norms_str"] = df["extracted_norms"].apply(lambda x: "; ".join(x))

# Most cited norms
all_norms = df["extracted_norms"].explode().dropna()
if len(all_norms) > 0:
    print("Top 20 cited norms:")
    print(all_norms.value_counts().head(20))
else:
    print("No norms extracted (full texts may not have been fetched yet).")
    print("The 'normen' column from the API metadata is also available:")
    print(df["normen"].value_counts().head(10))

Top 20 cited norms:
extracted_norms
§148 A             5
§ 229 StGB         5
§241e Abs          5
§ 241e Abs         5
§241e Absatz       4
§142 Da            4
§49 A              4
§127 F             4
§110 Abs           3
§28 Cb             3
§ 229 Abs          3
§28 Abs            2
§§ 28-31 Rz        2
§ 79 Abs           2
§49\nRechtssatz    2
§ 241e Rz          2
§ 129 Z            2
§ 127 StGB         2
§229 Abs           2
§ 49 EheG          2
Name: count, dtype: int64


In [38]:
# --- KWIC: Keyword in Context ---

def extract_kwic(text, keyword, window=200):
    contexts = []
    if not text:
        return contexts
    text_lower = text.lower()
    kw_lower = keyword.lower()
    start = 0
    while True:
        idx = text_lower.find(kw_lower, start)
        if idx == -1:
            break
        ctx_start = max(0, idx - window)
        ctx_end = min(len(text), idx + len(keyword) + window)
        contexts.append({
            "keyword": keyword,
            "position_in_text": idx,
            "context": text[ctx_start:ctx_end].strip(),
        })
        start = idx + 1
    return contexts

kwic_rows = []
primary_kw = ["Entfremdung", "Kindeswohl"]

for _, row in df.iterrows():
    for kw in primary_kw:
        for ctx in extract_kwic(row.get("full_text", ""), kw):
            kwic_rows.append({
                "id": row["id"],
                "geschaeftszahl": row["geschaeftszahl"][:50] if row.get("geschaeftszahl") else "",
                "gericht": row["gericht"],
                "year": row["year"],
                "dokumenttyp": row["dokumenttyp"],
                **ctx,
            })

df_kwic = pd.DataFrame(kwic_rows)
print(f"KWIC dataset: {len(df_kwic)} keyword occurrences")
if len(df_kwic) > 0:
    print(df_kwic["keyword"].value_counts())
    print(f"\nSample context:")
    print(df_kwic.iloc[0]["context"][:400])

KWIC dataset: 87 keyword occurrences
keyword
Entfremdung    73
Kindeswohl     14
Name: count, dtype: int64

Sample context:
e Ausübung des Besuchsrechtes durch den Vater drei Jahre hindurch verhindert hat, nun nicht darauf berufen, dass dieser drei Jahre hindurch keinen Kontakt zu seinem Kind gehabt habe und hiedurch eine Entfremdung zwischen dem Vater und seinem Kind eingetreten sei. (T2)
TE OGH 1982-10-05 5 Ob 733/82
TE OGH 1982-11-03 1 Ob 610/82
nur T1
TE OGH 1983-10-13 7 Ob 707/83
nur T1
TE OGH 1983-10-25 2 Ob 577/


## 6. Export

In [34]:
# --- Metadata CSV (no full text) ---
meta_cols = [c for c in df.columns if c not in ["full_text", "extracted_norms"]]
df[meta_cols].to_csv(OUTPUT_DIR / "decisions_metadata.csv", index=False, encoding="utf-8-sig")
print(f"✅ {OUTPUT_DIR / 'decisions_metadata.csv'}")

# --- Full dataset as Parquet ---
export_cols = [c for c in df.columns if c != "extracted_norms"]
df[export_cols].to_parquet(OUTPUT_DIR / "decisions_full.parquet", index=False)
print(f"✅ {OUTPUT_DIR / 'decisions_full.parquet'}")

# --- KWIC contexts ---
if len(df_kwic) > 0:
    df_kwic.to_csv(OUTPUT_DIR / "kwic_contexts.csv", index=False, encoding="utf-8-sig")
    print(f"✅ {OUTPUT_DIR / 'kwic_contexts.csv'}")

# --- Individual text files (for topic modeling) ---
texts_dir = OUTPUT_DIR / "texts"
texts_dir.mkdir(exist_ok=True)
count = 0
for _, row in df.iterrows():
    text = row.get("full_text", "")
    if not text or len(text) < 100:
        continue
    fn = re.sub(r"[^a-zA-Z0-9_-]", "_", row.get("id", "unknown")[:80])
    with open(texts_dir / f"{fn}.txt", "w", encoding="utf-8") as f:
        f.write(text)
    count += 1
print(f"✅ {count} text files → {texts_dir}/")

print(f"\n{'='*50}\nAll exports complete!")

✅ ris_data/decisions_metadata.csv


ImportError: Unable to find a usable engine; tried using: 'pyarrow', 'fastparquet'.
A suitable version of pyarrow or fastparquet is required for parquet support.
Trying to import the above resulted in these errors:
 - Missing optional dependency 'pyarrow'. pyarrow is required for parquet support. Use pip or conda to install pyarrow.
 - Missing optional dependency 'fastparquet'. fastparquet is required for parquet support. Use pip or conda to install fastparquet.

## 7. Visualizations

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Decisions per year
if df["year"].notna().any():
    df.groupby("year").size().plot(kind="bar", ax=axes[0, 0], color="steelblue")
    axes[0, 0].set_title("Documents per Year")
    axes[0, 0].tick_params(axis="x", rotation=45)

# 2. Court distribution
df["gericht"].value_counts().head(8).plot(kind="barh", ax=axes[0, 1], color="coral")
axes[0, 1].set_title("By Court")

# 3. Dokumenttyp
df["dokumenttyp"].value_counts().plot(kind="bar", ax=axes[1, 0], color="seagreen")
axes[1, 0].set_title("Document Type (Rechtssatz vs Text)")

# 4. Keyword presence
kw_cols = [c for c in df.columns if c.startswith("contains_")]
kw_summary = df[kw_cols].sum().sort_values(ascending=True)
kw_summary.index = [c.replace("contains_", "").replace("_", " ") for c in kw_summary.index]
kw_summary.plot(kind="barh", ax=axes[1, 1], color="mediumpurple")
axes[1, 1].set_title("Keyword Presence in Full Text")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "overview_plots.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {OUTPUT_DIR / 'overview_plots.png'}")

---

## Notes & Troubleshooting

### API Quirks
- The page size appears fixed at 20, regardless of `Seitengroesse` parameter
- `Seite` (page number) is 1-based
- Total hits is in `Hits["#text"]` as a string
- Single results come as a dict (not a list) — the parser handles this
- `Geschaeftszahl.item` can be a very long semicolon-separated string (many case numbers for one Rechtssatz)

### Rechtssatz vs Entscheidungstext
- **Rechtssatz** = legal principle summary, often cited across dozens of cases. Short but very information-dense. The `Geschaeftszahl` field lists all cases that reference this principle.
- **Text** (Entscheidungstext) = full decision text. Longer, contains the actual reasoning.
- For NLP, both are valuable but serve different purposes. Rechtssätze are great for understanding how courts define key concepts.

### Full text not found?
- Rechtssätze may not have a separate HTML content file. Their content is often the Rechtssatz itself.
- Check `content_url_html` — if empty, try `dokument_url` (the RIS website link).
- The `Entscheidungstexte.item[]` in the Justiz section links to the actual full decisions.

### Community Python wrapper
```python
pip install risApiWrapper
from risApiWrapper.Judikatur import Justiz
results = Justiz(search_words="Kindeswohl")
for decision in results:
    print(decision["case_number"])
```
See [github.com/PhilippTh/ris-API-wrapper](https://github.com/PhilippTh/ris-API-wrapper)

### Next steps for NLP
- `decisions_full.parquet` → BERTopic with `paraphrase-multilingual-MiniLM-L12-v2`
- `kwic_contexts.csv` → targeted analysis of keyword usage contexts
- The `normen` column (from API metadata) already lists cited laws — no text extraction needed
- Separate Rechtssätze and Entscheidungstexte for different analytical approaches